# Model 2 — 약관 조문을 실행 가능한 조건식으로

상품설명서 조문을 **기계가 실행할 수 있는 조건식 JSON** 변환,
그 조건식으로 고객별 최종 적용금리 계산.

```
약관 조문  ──▶  조건식 JSON  ──▶  개별 판별  ──▶  최종 적용금리
           태스크1          태스크2        태스크3
           (LLM)            (순수 코드)     (순수 코드)
```

**LLM 은 태스크 1 에만 쓴다.** 전처리(조문 분할)와 판별(3치 논리 실행기)은
전부 결정론적 코드다. 모델이 하는 일은 "조문 → 조건식" 매핑.

## 실험 대상

KB Young Youth 통장 상품설명서 — 조문 30개, 각 3회 생성해서 111 레코드.

---

## 0. 환경 확인

이 노트북은 저장소 루트에서 실행한다. `.py` 모듈이 전부 거기 평평하게 있고,
Jupyter 는 노트북이 있는 디렉터리를 `sys.path` 에 넣으므로 `import catalog` 가 그냥 된다.

**커널은 반드시 `.venv` 여야 한다.** 시스템 python 은 transformers 4.44 라
Qwen3 tokenizer 를 못 읽고 모델 로딩에서 죽는다. 아래 셀이 그걸 먼저 잡는다.

In [ ]:
# 저장소 루트. 옛 루트(/home/CYJ/M2-2)는 2026-09-11 컨테이너 재시작으로 사라졌다.
# 홈은 컨테이너 overlay(휘발성)라 저장소는 영구 볼륨 아래에 둔다.
import os, sys, json, platform
from pathlib import Path

REPO_ROOT = os.environ.get("M2_REPO_ROOT", "/workspace/personal_workplace_CHO/model/M2-2")

if not Path(REPO_ROOT, "catalog.py").is_file():
    print(f"저장소 없음: {REPO_ROOT}")
    print()
    print("  .py 모듈을 import 하는 셀(§1 이하)은 전부 돌지 않는다.")
    print("  복구한 뒤 위 경로에 두거나 M2_REPO_ROOT 로 다른 경로를 주면 된다.")
    print("  서술만 읽는 경우에는 이 셀을 건너뛴다 — 부록 A 참고.")
else:
    os.chdir(REPO_ROOT)
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

    print(f"python       {platform.python_version()}  ({sys.executable})")
    assert ".venv" in sys.executable, (
        "venv 커널이 아니다. 시스템 python 은 transformers 4.44 라 "
        "Qwen3 tokenizer.json 을 못 읽고 모델 로딩에서 죽는다.\n"
        "  .venv/bin/python -m ipykernel install --user --name m2-venv")

    import torch, transformers
    print(f"torch        {torch.__version__}")
    print(f"transformers {transformers.__version__}")

    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"GPU          {torch.cuda.get_device_name(0)}  "
              f"여유 {free/1e9:.1f} / {total/1e9:.1f} GB")
    else:
        print("GPU          없음 — §5(모델 실행)만 건너뛰면 나머지는 다 돈다")

    import catalog
    print(f"\n카탈로그     변수 {len(catalog.VARIABLES)} · 이벤트 {len(catalog.EVENTS)}"
          f" · 연산자 {len(catalog.OPERATORS)} · effect {len(catalog.EFFECTS)}")
    print(f"코퍼스       {len(json.loads(Path('data/clauses.json').read_text('utf-8')))} 조문")


---

## 1. 전처리 — 조문 분할 (LLM 없음)

**여기에 모델을 쓰지 않는다.** 상품설명서는 마크다운 표와 섹션이 있는 구조화된 문서라
규칙으로 자를 수 있고, 규칙으로 자르면 **매번 같은 결과가 나온다.**
전처리가 흔들리면 그 아래의 모든 측정이 흔들린다.

`doc_split.py` 가 하는 일:

| | |
|---|---|
| 섹션·표 인식 | `#` 헤딩과 마크다운 표를 블록으로 나눈다 |
| 페이지 분단 병합 | PDF 추출에서 페이지로 잘린 표를 같은 표로 다시 붙인다 |
| 카테고리 분류 | rate / benefit / eligibility / restriction / admin / tax |
| **참조표 인라인** | "아래 표 참조" 행에 그 표의 내용을 끌어와 붙인다 |
| **중복 제거** | 본문에 붙인 표는 단독 조문에서 뺀다 |

마지막 두 줄이 한 쌍이다.

모델이 표를 못 보면 조건식을 만들 수 없다. "가입대상: 아래 표 참조" 만 던져 주고
조건식을 기대할 수는 없으니 표를 본문에 끌어와 붙인다. 그런데 **붙인 표를 단독 조문으로도
남기면 같은 내용이 두 번 채점된다.**

실제로 그랬다. 08-27 라운드에서 표 7개가 전부 다른 조문에 완전히 포함돼 있었고,
111 레코드 중 21 건 — **코퍼스의 19%** 가 중복이었다. 평균을 조금 흐리는 정도가 아니라
**있는 회귀를 지우고 없는 개선을 만들었다.** 숫자는 §6.2 에서 다시 본다.

지금은 붙인 표를 단독 조문에서 뺀다. 코퍼스가 37 → 30 조문이 됐지만 **버린 내용은 없다.**
표는 본문 안에서 맥락과 함께 한 번 채점되고, 어느 본문도 부르지 않은 표는 그대로 단독
조문으로 남는다. 아래 셀이 찍는 `중복 : 없음` 이 그 불변식이고,
`tests/test_doc_split.py` 가 회귀를 막는다.

In [2]:
import collections
from doc_split import build, select

DOC = "data/KB Young Youth 통장 상품설명서.txt"

raw = build(DOC)                          # 규칙 기반 분할 — 결정론적
clauses = select(raw, skip="boilerplate")  # 예금자보호 안내 등 정형문구 제외

print(f"원문 {len(Path(DOC).read_text('utf-8')):,}자")
print(f"분할 {len(raw)}개 → boilerplate 제외 → {len(clauses)}개\n")

cat = collections.Counter(c["category"] for c in clauses)
ln = sorted(len(c["text"]) for c in clauses)
print("분류 :", dict(cat.most_common()))
print(f"길이 : 중앙 {ln[len(ln)//2]}자 (최소 {ln[0]}, 최대 {ln[-1]})")

# 조문끼리 서로를 포함하면 그 내용은 두 번 채점된다
dup = [(a["id"], b["id"]) for a in clauses for b in clauses
       if a is not b and a["text"] in b["text"]]
print("중복 :", dup or "없음")

saved = json.loads(Path("data/clauses.json").read_text("utf-8"))
print("\ndata/clauses.json 과 일치:", [c["id"] for c in clauses] == [c["id"] for c in saved])

원문 9,108자
분할 36개 → boilerplate 제외 → 30개

분류 : {'benefit': 8, 'eligibility': 6, 'restriction': 6, 'rate': 5, 'admin': 3, 'other': 1, 'tax': 1}
길이 : 중앙 108자 (최소 11, 최대 1546)
중복 : 없음

data/clauses.json 과 일치: True


### 조문 실물 — 참조표가 어떻게 붙는가

`KBYY-S5-R02` 는 참조표 2개를 흡수한 조문이다.
본문의 "아래 표 참조" 두 곳에 실제 표가 끌어와 붙었다.

표가 없으면 **"기본실적" 이 무엇인지 알 수 없어 조건식을 만들 수 없다.**
이것이 참조표를 인라인하는 이유다.

In [3]:
c = next(x for x in clauses if x["id"] == "KBYY-S5-R02")

print(f"[{c['id']}]  {c['label']}   ({c['category']}, {len(c['text'])}자, p{c['page']})")
print(f"인라인된 참조표: {c.get('refs')}")
print("-" * 78)
print(c["text"])

[KBYY-S5-R02]  KB생활 서비스Ⅰ   (benefit, 510자, p6)
인라인된 참조표: ['기본실적', '면제 대상 수수료 및 면제횟수']
------------------------------------------------------------------------------
[KB생활 서비스Ⅰ]
▪ “기본실적”만 있는 경우
➢ 면제조건 : 전월 중 이 통장에서 아래“기본실적 중 하나 이상” 발생된 경우 (아래 표 참조)
➢ 면제기간 : 이번 달 11일부터 다음달 10일까지
➢ 면제대상 수수료 및 면제횟수 (아래 표 참조)
※ 이 통장의 신규(또는 전환)일부터 다음 다음 달 10일까지는 은행 거래실적에 관계없이 면제

[참조표: 기본실적]
기본실적 | 실적 인정조건
KB카드결제 | KB국민카드(신용/체크/KB국민비씨 카드포함) 결제실적(현금서비스 포함)이 있는 경우
공과금이체 | 이 통장에서 각종 공과금이체(출금) 실적주)이 있는 경우
주) 아파트관리비, 지로, CMS, 펌뱅킹을 통한 자동납부(카드결제분 제외)

[참조표: 면제 대상 수수료 및 면제횟수]
면제 대상 수수료(이 통장 기준) | 면제횟수
전자금융(인터넷뱅킹, 폰뱅킹, 모바일뱅킹) 이체수수료(다른은행으로 보낼 때) | 제한없음
KB국민은행 자동화기기 시간외출금수수료 | 
납부자자동이체(타행자동이체포함)수수료 |


---

## 2. 카탈로그 — io 명세의 실체

**모델은 여기 있는 이름만 쓸 수 있다.** 벗어나면 실패로 처리한다(지표 ③).
자유 서술을 허용하면 같은 뜻이 열 가지 표기로 갈리고, 그러면 실행기를 만들 수 없다.

카탈로그를 늘릴지 말지가 이 프로젝트에서 반복되는 판단이다. 두 사례:

**`LOW_BALANCE` — 거부했다.** 모델이 "저금통 설정금액보다 실제 잔액이 더 적은 경우"를
`LOW_BALANCE` 라는 이벤트로 지어냈다. 그런데 이건 **사건이 아니라 상태 비교**다.
이벤트로 받아주면 모델링 오류가 카탈로그에 굳는다. 변수 두 개로 나눠 넣었다.

```
{"<": [{"var": "account.balance"}, {"var": "account.reserved_amount"}]}
```

**`FEE_WAIVE` — 추가했다.** 수수료 면제를 표현할 effect 가 없었는데, 모델은 포기하지 않고
`{"type": "LIMIT_AMOUNT", "value": 0}` (한도 0원) 으로 우회했다. 형식은 완벽해서
지표 ①~⑤ 를 전부 통과하는데 뜻은 정반대에 가깝다. **지표가 못 보는 실패였다** (§6).

프롬프트에는 카탈로그 전체가 아니라 **조문별 후보만** 주입한다.
전부 주면 프롬프트가 길어지고, 관계없는 이름이 오답 후보가 된다.

In [4]:
from catalog import VARIABLES, EVENTS, OPERATORS, EFFECTS, FEES, pick_candidates

print(f"변수 {len(VARIABLES)}개")
for n, m in list(VARIABLES.items())[:4]:
    print(f"   {n:<28}{m['type']:<7}{m['desc']}")
print(f"   ... 외 {len(VARIABLES)-4}개\n")

print(f"이벤트   {len(EVENTS):>2}개 :", ", ".join(list(EVENTS)[:6]), "...")
print(f"연산자   {len(OPERATORS):>2}개 :", ", ".join(list(OPERATORS)[:8]), "...")
print(f"effect   {len(EFFECTS):>2}개 :", ", ".join(EFFECTS))
print(f"면제대상 {len(FEES):>2}개 :", ", ".join(list(FEES)[:3]), "...")

print("\n조문별 후보 추림 — 프롬프트에는 이것만 들어간다")
for cid in ("KBYY-S2-R11", "KBYY-S5-R02"):
    c = next(x for x in clauses if x["id"] == cid)
    v, e = pick_candidates(c["text"])
    print(f"  [{cid}] {c['label']}")
    print(f"      변수   {v}")
    print(f"      이벤트 {e}")

변수 17개
   contract.start_date         date   가입일
   contract.maturity_date      date   만기일
   contract.term_months        int    가입기간(개월)
   contract.amount             int    가입금액(원)
   ... 외 13개

이벤트   10개 : SALARY_TRANSFER, AUTO_TRANSFER, CARD_USAGE, DEPOSIT, PENSION_RECEIPT, FX_TRANSACTION ...
연산자   14개 : >=, >, <=, <, ==, !=, and, or ...
effect    8개 : RATE_ADD, RATE_SUBTRACT, RATE_OVERRIDE, PENALTY_RATE, DISQUALIFY, LIMIT_AMOUNT, REQUIRE, FEE_WAIVE
면제대상  6개 : EBANK_TRANSFER, ATM_TRANSFER, ATM_WITHDRAW_AFTER_HOURS ...

조문별 후보 추림 — 프롬프트에는 이것만 들어간다
  [KBYY-S2-R11] 저금통 기능
      변수   ['account.balance', 'account.reserved_amount', 'contract.channel', 'contract.termination_type']
      이벤트 []
  [KBYY-S5-R02] KB생활 서비스Ⅰ
      변수   ['contract.channel', 'contract.monthly_amount', 'contract.reference_date', 'contract.termination_type', 'customer.is_first_deposit']
      이벤트 ['AUTO_TRANSFER', 'CARD_USAGE']


---

## 3. 프롬프트 조립

`SYSTEM_PROMPT` 는 규칙과 표기법, `build_user_prompt()` 는 few-shot 2건 + 변환할 조문 +
그 조문의 후보 변수를 붙인다.

프롬프트를 두 번 크게 고쳤고, 실측에서 **한 번은 크게 성공하고 한 번은 역효과**가 났다.

### 성공 — 노드 표기법 명시

모델이 `{"operator": ">=", "left": ..., "right": ...}` 같은 흔한 JSON 논리식 관용을
쓰고 있었다. 정본은 연산자 이름이 키다. "올바름 / 틀림" 쌍으로 적었더니

| 모델이 만들어낸 연산자 | 수정 전 | 수정 후 |
|---|---:|---:|
| `operator` / `left` / `right` / `op` / `operands` | 57건 | **0건** |

### 역효과 — `literal` 을 금지어로 불렀더니 늘었다

같은 절에 이렇게 적어 뒀었다.

```
틀림:  {"literal": 6}          <- 리터럴은 그냥 6 이라고 쓴다
```

| | 수정 전 | 수정 후 |
|---|---:|---:|
| `literal` | 4건 | **10건** |

**차이는 모델이 원래 쓰던 표현이냐다.** `operator`/`left`/`right` 는 흔한 관용이라
"그거 말고 이거"가 교정으로 작동했다. `literal` 은 원래 거의 안 쓰던 토큰인데,
금지하려고 이름을 꺼내는 순간 선택지로 각인된 것으로 보인다.

> **부정 예시는 모델이 이미 저지르는 실수에만 쓴다.**
> 안 하던 실수를 미리 막으려고 이름을 부르면 오히려 가르치는 셈이 된다.

지금은 `literal` 이라는 단어가 프롬프트에 없다. 긍정형으로만 적었고,
`tests/test_catalog.py` 가 `"literal" not in SYSTEM_PROMPT` 를 고정한다.

In [5]:
from prompt import SYSTEM_PROMPT, build_user_prompt
from transformers import AutoTokenizer

c = next(x for x in clauses if x["id"] == "KBYY-S5-R02")
user = build_user_prompt(c["text"])

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
conv = [{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user}]
text = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True,
                               enable_thinking=False)   # 매핑 태스크다. 추론 모드 끈다

print(f"SYSTEM_PROMPT  {len(SYSTEM_PROMPT):>6,}자")
print(f"user prompt    {len(user):>6,}자   few-shot 2건 + 조문 + 후보 변수")
print(f"합계           {len(tok(text)['input_ids']):>6,} 토큰")
print(f"생성 예산       3,072 토큰  (--max-new-tokens)")

assert "literal" not in SYSTEM_PROMPT, "금지어를 이름으로 부르면 오히려 각인된다"

SYSTEM_PROMPT   4,890자
user prompt     3,460자   few-shot 2건 + 조문 + 후보 변수
합계            3,555 토큰
생성 예산       3,072 토큰  (--max-new-tokens)


---

## 4. 출력 형태 — 조건식 DSL

모델이 뱉어야 할 JSON 이다. 층이 네 개다.

| 층 | 뜻 | 없으면 |
|---|---|---|
| `source.quote` | 근거가 된 **원문 그대로** | 환각을 잡을 수 없다 (지표 ④) |
| `let` | 기간 정의 — 이름을 붙여 재사용 | 같은 기간을 여러 곳에 복사하게 된다 |
| `guard` | 이 조항을 **볼지 말지** | 중도해지인데 우대금리를 계산한다 |
| `condition` | 충족 여부 | — |
| `effect` | 충족했을 때 일어나는 일 | — |

`guard` 와 `condition` 을 나눈 것이 핵심이다. "중도해지면 우대금리 없음"은
조건이 거짓인 게 아니라 **조항을 볼 필요 자체가 없는** 상태다.
둘을 섞으면 3치 논리에서 UNKNOWN 이 잘못 전파된다 (§7).

### `$` 의 비대칭 — 여기서 자주 틀린다

`let` 에 **묶을 때는 `window`(달러 없음)**, `{"ref": "$window"}` 로 **참조할 때만 `$`** 다.
프롬프트가 이 비대칭을 안 알려주고 "let 에서 먼저 정의해야 한다"고만 적어 뒀었다.

실제로 이 노트북을 쓰면서 프롬프트에 넣을 예시를 만들다가 **나도 `"let": {"$window": ...}`
로 틀렸고, 검증기에 태워서야 잡혔다.** 사람이 문서만 읽고 틀리는 형태면
모델이 틀리는 것도 당연하다. 지금은 전체 형태를 보여주고 비대칭을 명시한다.

In [6]:
from schema import FEWSHOT_CLAUSE, FEWSHOT_OUTPUT
from validate import check_catalog, check_executable, check_quote, _validator

rule = FEWSHOT_OUTPUT["rules"][0]
print("조문:", FEWSHOT_CLAUSE, "\n")
print(json.dumps(rule, ensure_ascii=False, indent=1))

# few-shot 이 틀린 형태를 가르치면 모델은 그대로 따라 한다. 그래서 검증기에 태운다.
print("\n검증  스키마", not list(_validator.iter_errors(FEWSHOT_OUTPUT)),
      "· 카탈로그", not check_catalog(rule),
      "· 실행", not check_executable(rule),
      "· 인용", not check_quote(rule, FEWSHOT_CLAUSE))

조문: 적금 가입월 1일부터 만기월의 전전월 마지막일까지 급여이체가 발생한 월이 총 6개월 이상인 경우 연 1.0%p의 우대금리를 제공합니다. 단, 중도해지 시에는 우대금리가 적용되지 않습니다. 

{
 "rule_id": "R-EX-01",
 "status": "OK",
 "label": "급여이체 우대금리",
 "source": {
  "quote": "급여이체가 발생한 월이 총 6개월 이상인 경우 연 1.0%p의 우대금리를 제공합니다"
 },
 "let": {
  "window": {
   "month_range": {
    "from": {
     "anchor": "contract.start_date",
     "offset_months": 0,
     "day": "first"
    },
    "to": {
     "anchor": "contract.maturity_date",
     "offset_months": -2,
     "day": "last"
    }
   }
  }
 },
 "guard": {
  "==": [
   {
    "var": "contract.termination_type"
   },
   "MATURITY"
  ]
 },
 "condition": {
  ">=": [
   {
    "count_distinct_months": {
     "event": "SALARY_TRANSFER",
     "within": {
      "ref": "$window"
     }
    }
   },
   6
  ]
 },
 "effect": {
  "type": "RATE_ADD",
  "value": 1.0,
  "unit": "%p",
  "applies_to": "BASE_RATE"
 }
}

검증  스키마 True · 카탈로그 True · 실행 True · 인용 True


---

## 5. 모델 실행  ⚠️ 여기만 GPU 가 필요하다

**이 절을 건너뛰어도 나머지는 전부 돈다.** 아래 §6 부터는 저장된 실험 결과
(`results/*.json`)를 읽으므로 GPU 없이 끝까지 실행된다.

아래 셀은 기본이 `RUN_MODEL = False` 다. `True` 로 바꾸면 조문 하나만 실제로 돌린다
(모델 로딩 약 80초 + 생성 약 25초). 전체 30조문 × 3회는 터미널에서 돌리는 게 낫다.

```bash
python run_local_v2.py --runs 3 --out results/results_kbyy_v3.json
```

### 이 서버(V100 32GB)의 제약

| | |
|---|---|
| bf16 불가 | sm70 이라 fp16 으로 돈다 |
| FlashAttention 불가 | SM80+ 필요. SDPA 로 돌면 어텐션이 O(n²) 로 메모리를 먹는다 |
| **vLLM 불가** | sm70 → TRITON_ATTN 백엔드 → triton 이 JIT 컴파일에 C 컴파일러를 요구하는데 컨테이너에 gcc 가 없다 |

vLLM 을 못 쓰면서 잃는 것은 `--guided`(constrained decoding) 하나인데,
**지표 ① JSON 파싱률이 제약 없이도 96% 라 처방할 병이 없다.**

### 왜 느린가 — 대역폭에 묶여 있다

batch 1 자기회귀 디코딩은 토큰 하나마다 16GB 가중치를 전부 읽는다.
V100 의 HBM 대역폭이 900GB/s 이므로 이론 상한이 초당 56토큰이고, 실제로는 약 20토큰이다.
**연산이 아니라 메모리 읽기가 병목이라 GPU 사용률을 올려도 빨라지지 않는다.**

그래서 같은 조문의 3회 생성을 `num_return_sequences` 로 한 배치에 묶는다.
가중치를 한 번 읽어 3개를 만드니 거의 3배 이득이다.

### 겪은 사고 두 개

**allocator 스래싱.** 초판은 메모리가 22.5GB → 31.7GB 로 불어나며 GPU 사용률이
57% → 3% 로 떨어지고 레코드당 70초 → 265초가 됐다.
`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` 로 해결했다.
**`import torch` 보다 먼저** 설정해야 한다.

**결과 44분치 유실.** 초판은 111건을 다 끝내야 파일을 썼다. 중간에 죽으니 전부 날아갔다.
지금은 조문마다 저장하고 `--resume` 으로 이어받는다. 실제로 이번 실행에서 OOM 으로
두 번 죽었고 108건에서 이어받아 완료했다.

In [7]:
RUN_MODEL = False   # True 로 바꾸면 조문 하나를 실제로 생성한다 (약 100초)

if RUN_MODEL:
    import time
    from run_local_v2 import load_model, run_clause

    demo = next(x for x in clauses if x["id"] == "KBYY-S3-R02")   # 29자짜리
    print("조문:", demo["text"].replace("\n", " "), "\n")

    t = time.time(); tok_m, model = load_model("Qwen/Qwen3-8B")
    print(f"모델 로딩 {time.time()-t:.0f}초")

    t = time.time()
    recs = run_clause(tok_m, model, demo, runs=1, temperature=0.1, allow_retry=True,
                      max_new_tokens=3072, batch_size=1)
    print(f"생성 {time.time()-t:.0f}초\n")

    r = recs[0]
    print("지표 :", {k: r["final"][k] for k in
                    ("json_ok", "schema_ok", "catalog_ok", "quote_ok", "exec_ok")})
    print("rule :", r["final"]["n_rules"], "· 시도", r["n_attempts"])
    print(json.dumps((r.get("output") or {}).get("rules", []), ensure_ascii=False, indent=1))
else:
    print("RUN_MODEL = False — 건너뛴다. §6 부터는 저장된 결과로 돈다.")

RUN_MODEL = False — 건너뛴다. §6 부터는 저장된 결과로 돈다.


---

## 6. 검증 — 무엇을 재는가

gold 조건식 없이도 잴 수 있는 것 다섯 가지다. `validate.py` 가 전부 계산한다.

| | 지표 | 묻는 것 |
|---|---|---|
| ① | `json_ok` | 코드펜스나 잡소리 없이 JSON 을 뱉는가 |
| ② | `schema_ok` | 4층 구조를 지키는가 |
| ③ | `catalog_ok` | 정의 안 된 이름을 만들어내는가 |
| ④ | `quote_ok` | 근거 인용을 지어내는가 |
| ⑤ | `exec_ok` | 실행기에 태웠을 때 ERROR 없이 끝나는가 |

검증기는 **첫 실패에서 멈추지 않고 모든 오류를 모아서** 반환한다.
하나만 보고하면 재시도 프롬프트가 나머지를 못 고친다.

⑤ 가 특별하다. **실행기가 곧 카탈로그 검증기를 겸한다** — 합성 ctx 로 조건식을
끝까지 걸어보므로, 스키마가 못 잡는 의미 오류(피연산자 개수, 타입, 미바인딩 참조)가 여기서 걸린다.

### 6.1 실측 — 2026-09-02 라운드

In [8]:
KEYS = ("json_ok", "schema_ok", "catalog_ok", "quote_ok", "exec_ok")
SYM, NAME = "①②③④⑤", ("JSON 파싱", "스키마 적합", "카탈로그 준수",
                        "quote 원문 일치", "실행 가능")

R = json.loads(Path("results/results_kbyy_v2.json").read_text("utf-8"))
print(f"레코드 {len(R)}건  (조문 {len({x['clause_id'] for x in R})}개 × 3회)\n")

print(f"{'지표':<22}{'1차':>8}{'재시도후':>10}{'개선':>8}")
print("-" * 50)
for i, k in enumerate(KEYS):
    a = sum(x["first"][k] for x in R) / len(R) * 100
    b = sum(x["final"][k] for x in R) / len(R) * 100
    print(f"{SYM[i]} {NAME[i]:<19}{a:>7.1f}%{b:>9.1f}%{b-a:>+7.1f}p")

att = [x["n_attempts"] for x in R]
print(f"\n생성 호출 {sum(att)}회 · 재시도 없이 통과 {att.count(1)}건"
      f" · 잘림 {sum(bool(x.get('truncated')) for x in R)}건")

레코드 111건  (조문 37개 × 3회)

지표                          1차      재시도후      개선
--------------------------------------------------
① JSON 파싱               96.4%     96.4%   +0.0p
② 스키마 적합                91.9%     94.6%   +2.7p
③ 카탈로그 준수               78.4%     87.4%   +9.0p
④ quote 원문 일치           88.3%     85.6%   -2.7p
⑤ 실행 가능                 40.5%     76.6%  +36.0p

생성 호출 211회 · 재시도 없이 통과 45건 · 잘림 3건


재시도가 ⑤ 를 **+36.0p** 끌어올린다. 자기수정 루프가 이 프로젝트에서 가장 크게 기여하는 부분이고,
왜 그런지는 §6.3 에서 다룬다.

④ 만 재시도로 **떨어진다**(−2.7p). 재시도가 조건식을 고치면서 인용을 다시 쓰기 때문이다.

> ⚠️ **위 표는 37조문 기준이고 오염돼 있다.** 참조표가 두 번 채점되고 있었다.
> 정정본은 §6.2 다.

### 6.2 숫자를 그대로 믿으면 안 되는 두 가지

이번 라운드에서 지표가 두 번 거짓말을 했다. 둘 다 모델 문제가 아니라 **측정 문제**다.

#### (가) 채점기를 함께 고쳤다 — 통제군이 필요하다

`validate.py`(정규화)와 `catalog.py`(이름 추가)를 모델 개선과 **같은 라운드에** 고쳤다.
그래서 8/27 숫자와 9/2 숫자를 그냥 빼면 **모델이 좋아진 몫과 채점기가 관대해진 몫이 섞인다.**

해결: 베이스라인의 **저장된 출력을 현재 검증기로 다시 채점**한다.
같은 모델 출력 · 같은 채점기이므로, 여기서 남는 차이는 전부 모델 몫이다.

#### (나) 참조표가 두 번 채점되고 있었다

표 7개가 전부 본문 조문에 **통째로 포함**돼 있었다. 코퍼스의 19% 가 같은 내용을 두 번 채점했다.
상수 편향이면 그나마 나았을 텐데 **라운드마다 방향이 뒤집혔다** —
8/27 에는 표가 본문보다 나쁘게(① −14p), 9/2 에는 좋게(⑤ 95% vs 본문 72%) 나왔다.

아래 셀이 둘 다 적용한 정정본을 계산한다.

In [9]:
import re
import validate as V

istab = lambda cid: bool(re.search(r"-T\d+$", cid))   # 표 단독 조문
TXT = {x["id"]: x["text"] for x in
       json.loads(Path("data/clauses.with-dup-tables.json").read_text("utf-8"))}

def rescore(records):
    """저장된 output 을 '현재' 검증기로 다시 채점한다."""
    c = {k: 0 for k in KEYS}
    for x in records:
        c["json_ok"] += x["final"]["json_ok"]
        c["schema_ok"] += x["final"]["schema_ok"]     # 스키마는 안 건드렸다
        rs = (x.get("output") or {}).get("rules")
        if not isinstance(rs, list) or not rs:
            for k in ("catalog_ok", "quote_ok", "exec_ok"):
                c[k] += x["final"][k]
            continue
        rr = [r for r in rs if isinstance(r, dict)]
        c["catalog_ok"] += not any(V.check_catalog(r) for r in rr)
        c["quote_ok"]   += not any(V.check_quote(r, TXT[x["clause_id"]]) for r in rr)
        c["exec_ok"]    += not any(V.check_executable(r) for r in rr)
    return {k: v / len(records) * 100 for k, v in c.items()}

B = json.loads(Path("results/results_kbyy.baseline.json").read_text("utf-8"))
body = lambda R: [x for x in R if not istab(x["clause_id"])]
print(f"표 조문 제외: {len(B)} → {len(body(B))} 레코드 "
      f"(중복 {len(B)-len(body(B))}건, {(len(B)-len(body(B)))/len(B)*100:.0f}%)\n")

ctrl, new = rescore(body(B)), rescore(body(R))          # 정정본 — 본문 30조문
dirty_b, dirty_n = rescore(B), rescore(R)               # 오염본 — 37조문

print(f"{'':<14}{'8/27':>8}{'9/2':>8}{'모델 몫':>9}   {'오염된 값':>10}")
print("-" * 54)
for i, k in enumerate(KEYS):
    print(f"{SYM[i]} {k:<12}{ctrl[k]:>7.1f}%{new[k]:>7.1f}%"
          f"{new[k]-ctrl[k]:>+8.1f}p{dirty_n[k]-dirty_b[k]:>+10.1f}p")

표 조문 제외: 111 → 90 레코드 (중복 21건, 19%)



                  8/27     9/2     모델 몫        오염된 값
------------------------------------------------------
① json_ok       100.0%   95.6%    -4.4p      -0.9p
② schema_ok      96.7%   93.3%    -3.3p      +0.0p
③ catalog_ok     85.6%   85.6%    +0.0p      +4.5p
④ quote_ok       96.7%   88.9%    -7.8p      -6.3p
⑤ exec_ok        42.2%   72.2%   +30.0p     +34.2p


**읽는 법이 달라진다.**

- **⑤ 는 여전히 큰 성공이다.** +30.0p. 4.2p 작아졌을 뿐 결론은 그대로다.
- **③ 의 개선은 사라졌다.** +4.5p 는 전부 표에서 나온 것이고 본문에서는 정확히 0 이다.
- **② 의 −3.3p 회귀는 아예 보이지 않았다.** 표가 가리고 있었다.
- ① 의 회귀도 −0.9p 가 아니라 −4.4p 로 4배 크다.

> 중복은 평균을 흐리는 데 그치지 않는다. **있는 회귀를 지우고 없는 개선을 만든다.**

지금은 `doc_split.py` 가 본문에 붙인 표를 단독 조문에서 뺀다(§1 의 `중복 : 없음`).
표를 버리는 게 아니라 본문 안에서 **맥락과 함께 한 번** 채점된다.

### 6.3 ⑤ 가 왜 30p 올랐나 — 오류 메시지가 프롬프트의 일부다

작업 전에 저장된 출력으로 효과를 추정했다. 표기법 오류를 기계적으로 교정해
"프롬프트만 고치면 얻는 상한"을 쟀고 **+3.6p** 가 나왔다. 실측은 **+30.0p** — **9배** 차이다.

추정이 놓친 것은 **재시도 루프의 성격 변화**다.
초판 `engine.py` 는 잘못된 조건식을 만나면 이렇게 죽었다.

```
AttributeError: 'dict' object has no attribute 'append'
```

이 메시지가 재시도 프롬프트에 그대로 들어갔다. 모델이 이걸 읽고 고칠 수 있을 리 없다.
방어 코드를 넣어 **무엇이 틀렸고 올바른 형태가 무엇인지** 말하게 바꿨다.
아래 셀이 지금 메시지를 보여준다.

정적 분석은 "이 출력이 통과하는가"만 본다. 하지만 실제로 일어난 일은
**모델이 무엇을 고쳐야 하는지 읽을 수 있게 된 것**이었다.
저장된 출력으로는 이 효과를 측정할 방법이 없다.

> **self-refinement 루프에서 오류 메시지는 프롬프트의 일부다.**
> 메시지 품질 개선은 저장된 출력으로 측정할 수 없고, 효과는 정적 추정보다 훨씬 클 수 있다.

In [10]:
import collections
from validate import check_executable

# and 의 값으로 배열이 아니라 객체를 준 경우 — 실제로 모델이 저지르던 실수
broken = {"rule_id": "R", "source": {"quote": "x"},
          "condition": {"and": {"a": {"==": [1, 1]}}},
          "effect": {"type": "RATE_ADD", "value": 1}}
print("잘못된 조건식 :", json.dumps(broken["condition"], ensure_ascii=False))
print("초판 메시지   : AttributeError: 'dict' object has no attribute 'append'")
print("지금 메시지   :", check_executable(broken)[0].replace("실행 불가: DslError: ", ""))

B = json.loads(Path("results/results_kbyy.baseline.json").read_text("utf-8"))
print()
for tag, X in (("8/27", B), ("9/2 ", R)):
    f = sum(x["first"]["exec_ok"] for x in X) / len(X) * 100
    l = sum(x["final"]["exec_ok"] for x in X) / len(X) * 100
    print(f"[{tag}] ⑤ 1차 {f:>4.1f}% → 재시도후 {l:>4.1f}%  (+{l-f:>4.1f}p)"
          f" · 생성 호출 {sum(x['n_attempts'] for x in X)}회")

print()
for tag, X in (("8/27", B), ("9/2 ", R)):
    c = collections.Counter()
    for x in X:
        for e in x.get("errors") or []:
            m = re.match(r"정의되지 않은 연산자: (.+)", e)
            if m:
                c[m.group(1).strip()] += 1
    print(f"[{tag}] 지어낸 연산자 {sum(c.values()):>3}건  {dict(c.most_common(5))}")

잘못된 조건식 : {"and": {"a": {"==": [1, 1]}}}
초판 메시지   : AttributeError: 'dict' object has no attribute 'append'
지금 메시지   : and: 배열을 기대했으나 dict 이 옴. 올바른 형태: {"and": [ {...}, {...} ]}

[8/27] ⑤ 1차 31.5% → 재시도후 42.3%  (+10.8p) · 생성 호출 257회
[9/2 ] ⑤ 1차 40.5% → 재시도후 76.6%  (+36.0p) · 생성 호출 211회

[8/27] 지어낸 연산자  61건  {'operator': 25, 'left': 10, 'right': 10, 'op': 9, 'literal': 4}
[9/2 ] 지어낸 연산자  17건  {'literal': 10, 'in': 3, 'month': 2, 'year': 2}


재시도 기여가 **+10.8p → +36.0p** 로 커졌고, 그러면서 **생성 호출은 257 → 211 회로 줄었다.**
메시지가 좋아지니 한 번에 고쳐지고, 최대 재시도까지 가는 경우가 줄었다.

지어낸 연산자도 61건 → 17건이다. 다만 `literal` 만 4 → 10 으로 **늘었다** —
§3 의 부정 예시 역효과가 여기서 관측된다.

### 6.4 지표가 못 보는 실패 — ①~⑤ 는 형식만 잰다

여기까지의 다섯 지표는 전부 **"형식이 맞는가"** 다.
스키마를 지켰나, 정의된 이름만 썼나, 크래시 없이 도나.
**"약관의 뜻을 옳게 옮겼나" 는 한 번도 재지 않았다.**

그 한계가 이번 라운드에 세 번 드러났다.

**(1) 수수료 면제를 "한도 0원" 으로 적었다.** 표현할 effect 타입이 없자 모델은
포기하지 않고 우회했다. 형식은 완벽해서 ①~⑤ 를 전부 통과하는데,
`resolver.py` 에서는 `other_effects` 로 **조용히 버려진다.**

**(2) ④ 는 환각의 총량이 아니라 위치를 잰다.** 같은 27자짜리 `[상품유형]` 라벨 조문에서
8/27 은 effect 값을, 9/2 는 인용을 지어냈다. 둘 다 환각인데 ④ 는 앞의 것만 통과시켰다.

**(3) §5 에서 방금 나온 것.** "제공기간 : 가입일로부터 1년" 을
`{"type": "RATE_OVERRIDE", "value": 12, "unit": "개월"}` — **금리를 12 로 대체**한다고 적었다.
지표 다섯 개 전부 통과다.

In [11]:
# (1) 수수료 면제에 붙은 effect 타입
c, sample = collections.Counter(), None
for x in R:
    for r in (x.get("output") or {}).get("rules") or []:
        if not isinstance(r, dict):
            continue
        q = (r.get("source") or {}).get("quote", "")
        if "면제" in q or "수수료" in q:
            t = (r.get("effect") or {}).get("type")
            c[t] += 1
            if t == "LIMIT_AMOUNT" and sample is None and all(x["final"][k] for k in KEYS):
                sample = (x["clause_id"], q, r["effect"])

print("수수료·면제 인용에 붙은 effect:", dict(c.most_common()))
cid, q, eff = sample
print(f"\n[{cid}]  ①~⑤ 전부 통과")
print(f"  원문 : {q[:58]}")
print(f"  생성 : {json.dumps(eff, ensure_ascii=False)}   ← '면제' 가 아니라 '한도 0원'")

# (2) 같은 조문, 같은 환각, 다른 ④ 판정
print("\nKBYY-S2-R03  원문은 27자짜리 [상품유형] 라벨이다")
for tag, X in (("8/27", B), ("9/2 ", R)):
    r = [x for x in X if x["clause_id"] == "KBYY-S2-R03"][0]
    y = ((r.get("output") or {}).get("rules") or [{}])[0]
    print(f"  [{tag}] ④={str(r['final']['quote_ok']):<5} "
          f"quote={(y.get('source') or {}).get('quote','')[:26]!r}")
    print(f"  {'':<7} {'':<9} effect={json.dumps(y.get('effect'), ensure_ascii=False)}")

수수료·면제 인용에 붙은 effect: {'LIMIT_AMOUNT': 36, 'DISQUALIFY': 9, 'RATE_ADD': 7, 'REQUIRE': 3, 'RATE_OVERRIDE': 2}

[KBYY-S2-T01]  ①~⑤ 전부 통과
  원문 : 전자금융(인터넷뱅킹, 폰뱅킹, 모바일뱅킹) 이체수수료(다른 은행으로 보낼 때) | 제한 없음
  생성 : {"type": "LIMIT_AMOUNT", "value": 0, "unit": "times"}   ← '면제' 가 아니라 '한도 0원'

KBYY-S2-R03  원문은 27자짜리 [상품유형] 라벨이다
  [8/27] ④=True  quote='입출금이 자유로운 예금(저축예금)'
                    effect={"type": "LIMIT_AMOUNT", "value": 50000000, "unit": "원"}
  [9/2 ] ④=False quote='중도해지 시에는 우대금리가 적용되지 않습니다'
                    effect={"type": "DISQUALIFY"}


> **①~⑤ 만으로는 이 실패들을 하나도 잡을 수 없다.**
> 실행 결과를 비교하는 지표 ⑥ 이 필요한 이유가 여기에 있다 (§8).

### 6.5 2026-09-03 라운드 — 통제군을 붙여 다시 쟀다

9/2 이후 손댄 것이 다섯 가지였다. `FEE_WAIVE` 카탈로그 추가, ④ 처방, 표기법,
긴 조문 분할, `day` 숫자 허용. **전부 미검증인 채로 쌓여 있었다.**

§6.2 가 말한 상황이 그대로다 — 채점기와 프롬프트를 같이 고쳐놓고 다음 숫자를
믿으려 하면 안 된다. 그래서 이번엔 코퍼스에 **통제군을 하나 심어서** 돌렸다.

| arm | 조문 | 무엇을 가리나 |
|---|---|---|
| 공통 29조문 | 9/2 와 동일 | 프롬프트·카탈로그 변경의 효과 |
| `R03a` `R03b` `R03c` | 분할군 | 조문 분할의 효과 |
| `R03-WHOLE` | 통짜 통제군 | 분할이 없었을 때의 같은 조문 |

한 번의 실행 안에 두 arm 이 같이 들어가야 한다. 따로 돌리면 실행 조건이
달라져서 차이가 분할 때문인지 그날 컨디션 때문인지 못 가린다.

In [ ]:
from pathlib import Path
import json, collections
from analyze import rate

V3 = json.loads(Path("results/results_kbyy_v3.json").read_text("utf-8"))
KEYS = (("json_ok", "① JSON 파싱"), ("schema_ok", "② 스키마 적합"),
        ("catalog_ok", "③ 카탈로그 준수"), ("quote_ok", "④ quote 일치"),
        ("exec_ok", "⑤ 실행 가능"))

print(f"9/3 라운드 — {len(V3)}건 · 조문 {len({x['clause_id'] for x in V3})}개 "
      f"· 잘림 {sum(1 for x in V3 if x.get('truncated'))}건\n")
print(f"{'지표':<16}{'1차':>8}{'재시도후':>10}{'개선':>8}")
for k, label in KEYS:
    a, b = rate(V3, "first", k), rate(V3, "final", k)
    print(f"{label:<16}{a:>7.1f}%{b:>9.1f}%{b - a:>+8.1f}")

# 9/2 대비 — 양쪽에 다 있는 (조문, run) 만 비교한다.
# R03 은 분할로 id 가 갈렸으므로 자동으로 빠진다.
V2 = json.loads(Path("results/results_kbyy_v2.json").read_text("utf-8"))
key = lambda x: (x["clause_id"], x["run"])
common = {key(x) for x in V2} & {key(x) for x in V3}
a2 = [x for x in V2 if key(x) in common]
a3 = [x for x in V3 if key(x) in common]

print(f"\n9/2 대비 (공통 {len(common)}건)")
print(f"{'지표':<16}{'9/2':>8}{'9/3':>8}{'변화':>8}")
for k, label in KEYS:
    a, b = rate(a2, "final", k), rate(a3, "final", k)
    print(f"{label:<16}{a:>7.1f}%{b:>7.1f}%{b - a:>+8.1f}")

**두 가지가 판정됐다.**

`FEE_WAIVE` — 9/2 에 `KBYY-S5-R03` 이 3회 모두 잘렸던 원인이 이것이었다.
분할 때문이 아니다. 통짜 조문을 **새 카탈로그로만** 돌려보니 잘림이 사라졌다.
반복 루프의 정체는 길이가 아니라 **수수료 면제를 표현할 effect 가 없어서**
`DISQUALIFY` 조합을 무한히 나열한 것이었다. 표현 수단을 주자 멈췄다.

④ 처방 — quote 일치가 **92.0% → 100.0%** 다. 처방이 먹었다.

⑤ 는 −2.3p 인데 조문별로 방향이 갈린다. `S2-R11`·`S2-R12` 는 크게 올랐고
`S2-R07`·`S2-R09`·`S2-R18`·`S2-R23` 은 내렸다. 상수 편향이 아니다.

In [ ]:
# 분할군 vs 통짜군 — 같은 조문, 같은 실행, 다른 전처리
SPLIT, WHOLE = {"KBYY-S5-R03a", "KBYY-S5-R03b", "KBYY-S5-R03c"}, {"KBYY-S5-R03-WHOLE"}

for tag, ids in (("분할군", SPLIT), ("통짜군", WHOLE)):
    rs = [x for x in V3 if x["clause_id"] in ids]
    got = [f"{label[0]} {rate(rs, 'final', k):.0f}%" for k, label in KEYS]
    print(f"{tag} ({len(rs)}건)  " + "  ".join(got))

# 형식 지표가 동점이면 값을 봐야 한다.
# 면제횟수 표의 정답 — 이체 월10회, 타행ATM 월5회, 나머지 제한없음.
TRUTH = {"EBANK_TRANSFER": "UNLIMITED", "ATM_WITHDRAW_AFTER_HOURS": "UNLIMITED",
         "AUTO_DEBIT": "UNLIMITED", "ATM_TRANSFER": 10,
         "OTHER_BANK_ATM_WITHDRAW": 5, "SMS_NOTICE": "UNLIMITED"}

print(f"\n면제횟수 6행을 맞히나 (같은 타깃에 rule 을 두 번 내면 '중복')")
for tag, ids in (("분할군", SPLIT), ("통짜군", WHOLE)):
    for run in (0, 1, 2):
        got = collections.defaultdict(list)
        for x in V3:
            if x["clause_id"] in ids and x["run"] == run:
                for r in (x.get("output") or {}).get("rules") or []:
                    e = r.get("effect") or {}
                    if e.get("type") == "FEE_WAIVE" and e.get("target"):
                        got[e["target"]].append(e.get("value"))
        ok = sum(1 for t, v in TRUTH.items() if got.get(t) and got[t][0] == v)
        dup = sum(1 for v in got.values() if len(v) > 1)
        print(f"  {tag} run{run}:  정답 {ok}/6 · 중복 {dup}")

**분할을 유지할 근거가 없다.**

①~⑤ 는 양쪽 다 100% 로 완전 동점이다. 값 정확도는 분할이 평균 5.33/6 對 5.0/6 으로
조금 앞서지만 **불안정하다** — 6, 4, 6 으로 흔들리고, 그 대가로 `R03a` 와 `R03c` 가
같은 fee 타깃에 rule 을 중복 생성한다 (중복 2, 6, 1 對 0, 0, 1).

분할이 풀려던 문제(잘림)는 `FEE_WAIVE` 가 이미 풀었다. 남은 이득은 측정 오차와
구분되지 않는 크기이고, 조문이 1개에서 3개로 늘면 **gold 라벨링 부담이 3배**가 된다.
⑥ 이 없는 지금 그건 실비용이다.

그래서 `doc_split.py` 의 분할은 **기본값을 꺼 두고** 코드와 테스트만 남긴다.
다른 문서에 진짜 긴 조문이 나오면 `--row-budget` 으로 켠다.

한 가지 덧붙일 것 — 여기서 "정답 6/6" 은 사람이 표를 보고 매긴 값이지
gold 기반이 아니다. **이 판정도 결국 ⑥ 으로 다시 확인해야 한다.**

---

## 7. 판별 — 조건식으로 실제 금리를 계산한다 (LLM 없음)

여기부터는 모델이 없다. 태스크 2·3 은 순수 코드다.

### 7.1 3치 논리 — 이 엔진의 유일한 비자명 부분

**고객 데이터가 '없는 것'은 false 가 아니다.**

급여이체 이력이 ctx 에 아예 없는 경우와, 있는데 5개월인 경우는 **다른 상태**다.
불리언으로 짜면 둘 다 NOT_MET 이 되어 **조용히 틀린 금리가 나간다.**
전자는 "판단 불가 — 사람이 봐야 함", 후자는 "조건 미충족" 이다.

그래서 Kleene strong 3치 논리를 쓴다. UNKNOWN 이 섞여도 결론이 나는 경우가 있다 —
`and` 에 거짓이 하나 있으면 나머지를 몰라도 거짓이고, `or` 에 참이 하나 있으면 참이다.
그 두 경우를 살리는 것이 Kleene 논리의 핵심이고, 실무에서는
**"모르는 항목이 있다고 무조건 사람에게 넘기지는 않는다"** 는 뜻이 된다.

In [12]:
from tv import TV, tv_and, tv_or, tv_not

vals = [TV.T, TV.F, TV.U]
for name, fn in (("and", tv_and), ("or", tv_or)):
    print(f"  {name:<4}" + "".join(f"{v.name:>6}" for v in vals))
    for a in vals:
        print(f"  {a.name:<4}" + "".join(f"{fn(a, b).name:>6}" for b in vals))
    print()

print("not U =", tv_not(TV.U).name, "\n")
print("모르는 값이 있어도 결론이 나는 두 경우")
print("  F and U =", tv_and(TV.F, TV.U).name, "  거짓이 하나 있으면 나머지는 볼 필요 없다")
print("  T or  U =", tv_or(TV.T, TV.U).name, "  참이 하나 있으면 나머지는 볼 필요 없다")

  and      T     F     U
  T        T     F     U
  F        F     F     F
  U        U     F     U

  or       T     F     U
  T        T     T     T
  F        T     F     U
  U        T     U     U

not U = U 

모르는 값이 있어도 결론이 나는 두 경우
  F and U = F   거짓이 하나 있으면 나머지는 볼 필요 없다
  T or  U = T   참이 하나 있으면 나머지는 볼 필요 없다


### 7.2 판별 실행 — 조건식 하나, 고객 넷

§4 의 few-shot 조건식("급여이체 6개월 이상이면 +1.0%p, 단 중도해지면 제외")을
고객 네 명에게 태운다. 네 명이 **네 가지 다른 상태**로 갈리는 것이 요점이다.

| 고객 | 기대 |
|---|---|
| 급여이체 6개월 | `MET` — 우대금리 적용 |
| 급여이체 3개월 | `NOT_MET` — 조건 미충족 |
| **급여이체 데이터 없음** | `UNKNOWN` — 판단 불가 |
| 중도해지 | `SKIPPED_GUARD` — 조항을 볼 필요 자체가 없음 |

3 번과 2 번을 구분하지 못하면 **데이터가 없는 고객에게 조용히 낮은 금리가 나간다.**
4 번을 `NOT_MET` 으로 뭉개면 "조건을 안 지켰다" 는 잘못된 사유가 남는다.

In [13]:
from engine import evaluate_rules
from resolver import resolve
from schema import FEWSHOT_OUTPUT
import gold

rules = FEWSHOT_OUTPUT["rules"]
prod = json.loads(Path("data/gold/_product.json").read_text("utf-8"))
agg, base = prod["aggregation"], prod["base_rate"]      # 상품설명서 실제 값

def ctx_of(급여개월, termination="MATURITY", 데이터없음=False):
    _, c = gold.profiles()[0]
    c = json.loads(json.dumps(c))                        # 깊은 복사
    c["contract"]["termination_type"] = termination
    if 데이터없음:
        # 빈 리스트 [] 가 아니라 _provided 에서 빼야 '미제공' 이 된다.
        c["events"].pop("SALARY_TRANSFER", None)
        c["_provided"] = [p for p in c["_provided"] if p != "events.SALARY_TRANSFER"]
    else:
        c["events"]["SALARY_TRANSFER"] = [
            {"date": f"2024-{m:02d}-25", "amount": 2_000_000} for m in range(1, 1 + 급여개월)]
    return c

cases = [("급여이체 6개월", ctx_of(6)),
         ("급여이체 3개월", ctx_of(3)),
         ("급여이체 데이터 없음", ctx_of(0, 데이터없음=True)),
         ("중도해지", ctx_of(6, termination="EARLY"))]

print(f"{'고객':<22}{'판별':<16}{'최종금리':>12}{'검토필요':>10}")
print("-" * 62)
for name, ctx in cases:
    res = evaluate_rules(rules, ctx)
    br = resolve(res, agg, base_rate=base)
    print(f"{name:<22}{res[0].status:<16}"
          f"{br.final_rate_min + '~' + br.final_rate_max:>12}{str(br.needs_review):>10}")

고객                    판별                      최종금리      검토필요
--------------------------------------------------------------
급여이체 6개월              MET                1.10~1.10     False
급여이체 3개월              NOT_MET            0.10~0.10     False
급여이체 데이터 없음           UNKNOWN            0.10~1.10      True
중도해지                  SKIPPED_GUARD      0.10~0.10     False


세 번째 줄이 이 설계의 값이다. 급여이체 데이터가 없으면 금리를 **구간 `0.10~1.10`** 으로 내고
`needs_review=True` 를 세운다. 하나의 숫자로 뭉개지 않는다 —
**"모르는 상태" 를 숫자로 감추면 그 순간 틀린 금리가 확정된다.**

금리 계산은 `Decimal` 로 하고 **반올림은 마지막에 한 번만** 한다.
중간에 반올림하면 조항이 여러 개일 때 오차가 누적된다.
상한(cap)은 상품설명서의 "최고 연 2.0%" 에서 왔다 — 기본금리 0.1% 를 뺀 **1.9%p** 다.

---

## 8. 지표 ⑥ — 뜻이 맞는지 재기

§6.4 에서 본 실패들은 ①~⑤ 로 하나도 잡히지 않는다. 그래서 gold 조건식이 필요하다.

### 구조가 아니라 결과를 비교한다

JSON 문자열이나 필드 일치율로 재면 잘못된 신호를 얻는다. 실측 근거가 있다.

- 구조 유사도가 높은(SP ≥ 0.5) 생성물 중 **36% 가 결과 일치도 50% 미만**
- 결과 일치도가 높은 것의 **39% 는 구조 유사도가 낮음**

`scoring.py` 는 gold 와 생성물을 **둘 다 실행해서** 결과를 비교한다.
rule 을 2개로 쪼개든 1개로 합치든, 고객에게 같은 금리가 나오면 같은 뜻이다.

### 라벨러 부담이 작다

실행기가 있으므로 **케이스별 정답(이 고객은 3.5%)을 만들 필요가 없다.**
조문당 조건식 하나만 쓰면 고객 프로필 조합은 도구가 만든다.

In [14]:
import copy
import gold
from scoring import score

prod = json.loads(Path("data/gold/_product.json").read_text("utf-8"))
AGG, BASE = prod["aggregation"], prod["base_rate"]
P = gold.profiles()
print(f"프로필 {len(P)}개 = " + " × ".join(f"{k}{len(v)}" for k, v in sorted(gold.AXES.items())))

BASE_RULE = {"rule_id": "R-01", "source": {"quote": "x"},
             "condition": {">=": [{"var": "customer.age"}, 19]},
             "effect": {"type": "RATE_ADD", "value": 0.5, "unit": "%p"}}

split = [{**BASE_RULE, "rule_id": f"R-0{i}",
          "effect": {"type": "RATE_ADD", "value": 0.25, "unit": "%p"}} for i in (1, 2)]
diff_value = copy.deepcopy(BASE_RULE); diff_value["effect"]["value"] = 1.5
waive = {**BASE_RULE, "effect": {"type": "FEE_WAIVE", "target": "EBANK_TRANSFER",
                                 "value": "UNLIMITED"}}
limit = {**BASE_RULE, "effect": {"type": "LIMIT_AMOUNT", "value": 0}}

print()
for desc, g, gen in [
    ("rule 1개 vs 2개로 쪼갬 (합 동일)", [BASE_RULE], split),
    ("구조 동일, 금리값만 0.5 vs 1.5",   [BASE_RULE], [diff_value]),
    ("수수료 면제 vs 한도 0원",          [waive],     [limit]),
    ("생성물이 빈 배열",                 [BASE_RULE], []),
]:
    print(f"{desc:<34}{score(g, gen, AGG, P, BASE)['rate']*100:>7.1f}%")

프로필 64개 = age2 × channel2 × termination2 × 급여2 × 실적2 × 저금통2

rule 1개 vs 2개로 쪼갬 (합 동일)            100.0%
구조 동일, 금리값만 0.5 vs 1.5               50.0%
수수료 면제 vs 한도 0원                      50.0%
생성물이 빈 배열                            50.0%


첫 줄이 **100%** 인 것이 요점이다. rule 을 몇 개로 쪼갰는지는 뜻이 아니다.

나머지가 **50%** 인 이유는 이 조건식이 `customer.age >= 19` 라서다.
age 17 인 프로필 절반에서는 rule 이 발동하지 않아 양쪽이 같은 결과를 낸다.
**차이는 rule 이 실제로 발동하는 프로필에서만 드러난다** — 그게 정상이다.

세 번째 줄이 §6.4 (1) 의 실패다. `FEE_WAIVE` 와 `LIMIT_AMOUNT 0` 은
①~⑤ 를 똑같이 통과하지만 **⑥ 은 구분한다.**

> 이 도구를 만들면서 `scoring.py` 의 구멍을 하나 찾았다.
> `outcome()` 이 금리·페널티·자격만 보고 `other_effects` 를 안 봐서
> **⑥ 조차 위 세 번째 줄을 통과시키고 있었다.** 지금은 포함한다 —
> 다만 `rule_id` 는 일부러 뺐다. 쪼갰는지 합쳤는지는 뜻이 아니기 때문이다.

### 8.2 라벨링 워크플로

```bash
python gold.py select   # 대상과 선정 근거
python gold.py init     # data/gold/ 에 템플릿 — 모델 출력이 초안으로 깔린다
#   ... 사람이 rules 를 고치고 reviewed 를 true 로 ...
python gold.py check    # 스키마·카탈로그·실행·원문 검증. 즉시 피드백
python gold.py score    # 지표 ⑥
```

**30조문을 다 할 필요는 없다.** 유형별로 15개면 방향이 보인다. 고른 기준:

| 왜 넣었나 | 예 |
|---|---|
| **①~⑤ 를 통과했지만 뜻이 의심스럽다** | 수수료 면제 조문 — 지표가 못 보는 실패가 여기 숨는다 |
| 빈 결과가 정답인지 | 3회 모두 빈 배열로 나온 조문 |
| 3회가 흔들린다 | rule 수 `[2, 0, 2]` — 셋 중 무엇이 맞나 |
| ⑤ 실패 | 정답을 알아야 고칠 방향이 선다 |
| 정상 기준점 | 이게 맞아야 나머지 비교가 의미 있다 |

첫 줄이 중요하다. **전통과 조문을 일부러 넣었다.** 실패한 것만 라벨링하면
"지표가 못 보는 실패" 를 영원히 못 본다.

`init` 이 모델 출력을 초안으로 깔아준다. 백지보다 빠르고, 무엇보다
**모델이 어디서 틀렸는지 보면서 고치게 된다.** 다만 그대로 두면
모델을 모델로 채점하는 꼴이라 `status` 가 손대지 않은 파일을 표시한다.

In [15]:
import gold

cl = {c["id"]: c for c in clauses}
tpl = {p.stem: json.loads(p.read_text("utf-8"))
       for p in sorted(Path("data/gold").glob("*.json")) if not p.name.startswith("_")}

print(f"대상 {len(gold.TARGETS)}개 / 전체 {len(cl)}조문 · "
      f"템플릿 {len(tpl)}개 · 검토완료 {sum(bool(v.get('reviewed')) for v in tpl.values())}\n")

print(f"{'조문':<15}{'분류':<12}{'자수':>5}  고른 이유")
print("-" * 92)
for cid, why in list(gold.TARGETS.items())[:6]:
    c = cl[cid]
    print(f"{cid:<15}{c['category']:<12}{len(c['text']):>5}  {why[:56]}")
print(f"{'... 외 ' + str(len(gold.TARGETS)-6) + '개':<15}")

대상 15개 / 전체 30조문 · 템플릿 15개 · 검토완료 0

조문             분류             자수  고른 이유
--------------------------------------------------------------------------------------------
KBYY-S5-R02    benefit       510  수수료 면제를 LIMIT_AMOUNT 0원 으로 적었는지 확인. ⑤ 통과했다
KBYY-S5-R01    benefit       263  수수료 면제 + rule 수 흔들림 [4,3,4]. 셋 중 무엇이 맞나
KBYY-S2-R03    eligibility    27  27자 [상품유형] 라벨인데 rule 2개를 만들었다. 빈 배열이 정답 아닌가
KBYY-S2-R02    eligibility    13  3회 모두 빈 배열. 정말 표현할 게 없는 조문인가
KBYY-S2-R16    restriction    11  3회 모두 빈 배열. 위와 같은 확인
KBYY-S2-R06    rate           84  rule 수 [2,0,2]. 0개가 맞나 2개가 맞나
... 외 9개       


---

## 9. 지금 어디까지 왔나

### 파이프라인

| | 단계 | 상태 |
|---|---|---|
| 태스크 1 | 조문 → 조건식 (LLM) | ✅ 실문서로 2라운드 실측 |
| 태스크 2 | 조건식 → 개별 판별 (`engine.py`) | ⚠️ **합성 ctx 로만** |
| 태스크 3 | 개별 판별 → 최종 금리 (`resolver.py`) | ⚠️ **손으로 만든 픽스처로만** |
| 태스크 4 | MODEL 3 용 OUTPUT 직렬화 | ❌ 미착수 |

**태스크 1 만 실제 문서로 돌려봤다.** `resolver.py` 는 테스트에서만 import 되고,
검증에 쓴 데이터는 손으로 만든 가상 상품이다.
**KB 문서에서 생성한 조건식이 resolver 를 통과한 적은 한 번도 없다.**
§7.2 에서 돌린 것도 few-shot 예시 조건식이지 실제 생성물이 아니다.

### 태스크 1 안에서

본문 30조문 기준, 다섯 라운드 모두 현재 검증기로 재채점한 값이다.

| 지표 | 8/27 | 9/2 | 9/3 (v3) | 9/3 (v4) | 9/4 (v5) |
|---|---:|---:|---:|---:|---:|
| ① JSON 파싱 | 100.0% | 95.6% | **100.0%** | 100.0% | 95.6% |
| ② 스키마 적합 | 96.7% | 93.3% | **96.7%** | 96.7% | 95.6% |
| ③ 카탈로그 준수 | 85.6% | 85.6% | **94.4%** | 95.6% | 93.3% |
| ④ quote 일치 | 96.7% | 88.9% | **100.0%** | 100.0% | 95.6% |
| ⑤ 실행 가능 | 42.2% | 72.2% | 73.3% | 73.3% | **90.0%** |
| ⑦ 빈 배열 (별도 축) | — | — | 10.3% | 10.0% | 13.3% |
| **⑥ 실행 결과 일치** | — | — | — | — | **49.3%** |

9/2 에 내줬던 ①②④ 를 9/3 에 되찾았고 ④ 는 두 라운드 연속 100% 였다 — NuExtract 판단은 끝났다.

**v4 까지 ⑤ 는 72~73% 에서 멈춰 있었다.** v4 는 재시도 오류 메시지만 바꾼 A/B 였는데
소수점까지 같았다. 그 대신 v4 가 **노이즈 바닥**을 재줬다 — 오류 메시지는 1차 생성에
영향을 줄 수 없으므로 1차 지표의 v3↔v4 차이가 곧 노이즈다. **90레코드에서 ±2~4p.**

**v5 가 그 벽을 넘었다.** few-shot 예시 구성을 바꾸자 ⑤ 가 **+14.2p** 다 (현재 계약).
재시도가 아니라 **1차 생성**을 건드린 첫 처방이고, 지금까지 판정 가능한 크기로 움직인
것은 이것뿐이다.

⑤ 만은 계약이 바뀌었으니 주의해서 읽어야 한다. 9/8 에 **두 번** 바뀌었다.

| 계약 | v3 | v4 | v5 | v4→v5 |
|---|---:|---:|---:|---:|
| ① 원래 (빈 배열 = 통과) | 72.4% | 73.3% | 90.0% | +16.7p |
| ② 빈 배열을 ⑦ 로 뗌 | 69.2% | 70.4% | 88.5% | +18.1p |
| **③ effect값 검사 추가** | **69.2%** | **70.4%** | **84.6%** | **+14.2p** |

②는 빈 배열이 ③④⑤ 를 공짜로 통과하던 것을 끊은 것이고, ③은 금리 effect 의 `value` 가
수가 아니면(`"contract.base_rate"` 같은 변수 이름) 실패로 친 것이다. 후자는 resolver 가
실제로 죽는 형태인데 세 라운드 내내 ⑤ 를 통과하고 있었다.

**어느 계약으로 읽어도 노이즈 바닥(±2~4p)의 3배 이상이다.** 앞으로의 비교는 ③ 열로 한다.

①과 ④가 −4.4p 로 내렸는데 **노이즈 바닥 경계선이라 판정하지 않는다.**

**⑥ 을 9/8 에 처음 쟀다 — 49.3%** (gold 15조문 × 프로필 128개 × 3회).
그런데 이 값을 그대로 인용하면 안 된다. **gold 15개 중 13개가 빈 배열**이라
분모의 대부분이 "모델도 가만히 있었나" 를 잰다. 조건식이 있는 두 조문이 본체다.

| 조문 | ⑥ | 무엇이 틀렸나 |
|---|---:|---|
| `S2-R07` 특별금리 | **33.3%** | `RATE_ADD` 를 `RATE_OVERRIDE` 로 — 더할 것을 덮어썼다. 2.00% 대 1.90% |
| `S5-R02` 수수료면제 | **16.7%** | 시간외출금수수료를 이체수수료로, "제한없음" 을 10회로 |

`S2-R07` 은 effect 타입도 카탈로그에 있는 것이고 수도 원문의 1.9 그대로다.
틀린 것은 **`ADD` 와 `OVERRIDE` 한 단어**이고, 그 결과가 10bp 다.
**①~⑤ 는 이것을 전부 통과시켰다.**

### 이번 라운드에서 배운 것

1. **self-refinement 루프에서 오류 메시지는 프롬프트의 일부다.**
   정적 추정 +3.6p vs 실측 +30.0p — 9배 차이가 여기서 났다.
2. **부정 예시는 모델이 이미 저지르는 실수에만 쓴다.**
   안 하던 실수를 미리 막으려고 이름을 부르면 오히려 가르친다 (`literal` 4 → 10건).
3. **지표 오염은 평균을 흐리는 데 그치지 않는다.**
   있는 회귀를 지우고 없는 개선을 만든다 (참조표 중복).
4. **형식 지표만으로는 환각을 못 잡는다.**
   ④ 는 환각의 총량이 아니라 위치를 잰다.

세 번은 **모델이 아니라 측정이 문제**였다. 지금 숫자는 8/27 보다 훨씬 믿을 만하다.

9/3 라운드가 여기에 하나를 더했다.

5. **지표가 제자리라고 아무 일도 없었던 게 아니다.**
   ⑤ 는 +1.1p 였지만 실패 조문의 절반이 교체됐다. 프롬프트를 고치면 다른 데가 터진다 —
   30조문 × 3회로는 이 폭을 신호로 읽으면 안 된다.
6. **노이즈 바닥을 모르면 개선을 주장할 수 없다.**
   재시도에만 영향을 주는 변경은 1차 지표가 공짜 통제군이 된다. v4 가 이 방법으로
   ±2~4p 를 쟀고, 그러자 **§10 의 "+1.1p 효과"가 판정 불가로 바뀌었다.**
7. **남은 ⑤ 실패는 흔들림이 아니라 벽이다.**
   6개 조문이 두 라운드 · 18회 시도를 못 넘었고, 20개 조문은 두 라운드 모두 100% 다.
   ⑤ 의 73% 는 **3분의 2는 항상 되고 나머지는 항상 안 된다**는 뜻이다.

v5 가 여기에 두 개를 더 얹었다.

8. **벽은 재시도가 아니라 예시로 넘었다.**
   오류 메시지 처방은 v4 에서 0 이었는데, few-shot 예시 구성을 바꾸자 +18.1p 다.
   실패를 사후에 고치는 것보다 **처음부터 다른 형태를 보여주는 쪽**이 셌다.
9. **채점 계약이 느슨하면 개선의 크기를 못 믿는다.**
   빈 배열이 ③④⑤ 를 공짜로 통과하던 동안 v5 의 이득은 16.7p 로 보였다.
   빈 배열을 축으로 떼면 18.1p 다 — 이번엔 실물이 더 컸지만,
   **방향이 반대였어도 알 방법이 없었다.**
10. **지시문이 있어도 예시가 없으면 안 된다.**
   규칙 5 는 처음부터 "표현할 수 없으면 생략하라" 였는데, gold 기준 빈 배열 13조문 중
   10개에서 모델이 rule 을 만들었다. 예시 1~3 이 전부 rule 을 내놓는 형태라
   **모델이 본 빈 배열이 하나도 없었다.** §12.2 와 정확히 같은 자리다.
11. **새 탈출구를 열 때는 채점을 먼저 막는다.**
   `UNSUPPORTED` 는 스키마에 처음부터 있었고 엔진도 처리했는데, 그냥 가르쳤으면
   어려운 조문마다 붙여 ⑤ 를 공짜로 올릴 수 있었다 — 실측으로 ①~⑤ 전부 통과, 오류 0건.
12. **지표로 못 가르는 것을 형태로는 가른다.**
   ⑤ 하나로는 한 라운드에 넣은 변경 둘을 못 나눈다. 틀린 형태를 각각 세면
   지표가 안 움직여도 어느 처방이 먹었는지 나온다 — v5 는 **둘 다** 먹었다
   (`{"var":...}` 단독 15→6, 미바인딩 `$window` 15→7). `analyze.py forms`.

### 남은 작업

| | 할 일 | 누가 |
|---|---|---|
| 1 | ~~`data/gold/*.json` 15개 채우기~~ → **완료 (9/8). ⑥ = 49.3%.** 15/15 검토, 검증 실패 0 | |
| 1-a | **gold 를 30조문으로 넓히기** — 13/15 가 빈 배열이라 ⑥ 의 분모 대부분이 "모델이 가만히 있었나" 를 잰다 | **팀 — 도메인 판단** |
| 1-b | **카탈로그 확장 4건** — 상품 전환 effect · 적용 기준 금액 effect · 압류 계열 이벤트 · 출금 이벤트 | **팀 — io 명세 합의** |
| 2 | ~~미검증 변경 3건 검증~~ → **완료 (9/3).** ④ 처방 ✅ · `FEE_WAIVE` ✅ · 표기법 ⚠(+1.1p) | |
| 3 | `FEE_WAIVE` 6종 · `LOW_BALANCE` 거부 검토 | **팀 — io 명세 합의** |
| 4 | ~~`KBYY-S5-R03` 분할~~ → **불필요.** 9/3 에 미분할로 3회 전부 통과, 잘림 0 | |
| 5 | `exclusive_groups` 를 비금리 effect 로 확장 | |
| 6 | few-shot 에 `FEE_WAIVE "value": "UNLIMITED"` 예시 — "제한없음" 우회 8건 | |
| 7 | ~~v4 실행~~ → **완료, 효과 없음.** 재시도 메시지 처방은 여기까지 | |
| 8 | ~~few-shot 예시 구성 손보기~~ → **완료, 크게 먹었다 (9/4).** ⑤ +18.1p | |
| 9 | 두 번째 상품설명서 추가 — 노이즈 바닥 ±2~4p 아래는 측정이 안 된다 | |
| 10 | ~~검증기의 빈 배열 통과 막기~~ → **완료 (9/8).** 3번째 축 ⑦ 로 뗐다 | |
| 11 | ~~v5 귀속~~ → **완료 (9/8).** `analyze.py forms` — **두 처방이 거의 똑같이 먹었다** (15→6, 15→7) | |
| 12 | ~~`UNSUPPORTED` 를 프롬프트에 가르치기~~ → **완료 (9/8).** 규칙 5 를 두 경우로 갈랐고 예시 4(빈 배열) 추가. 가르치기 전 검증기부터 막았다 — `UNSUPPORTED` 하나만 낸 출력이 ①~⑤ 를 전부 통과했다. **다음 실행 대기** | |
| 13 | ① · ④ −4.4p 확인 — 노이즈 경계선이라 판정 보류. ④ 는 두 라운드 연속 100% 였다 | |
| 14 | ~~⑥ 채점 파이프라인 점검~~ → **완료 (9/8).** 버그 2건 고쳤다 — 안 고쳤으면 팀이 라벨링을 끝낸 뒤에야 죽는 걸 봤다 | |
| 15 | ~~`RATE_*` 의 `value` 에 변수 이름~~ → **완료 (9/8).** `check_effect_value` 로 막았다. ⑤ 계약이 또 바뀌었다 | |
| 16 | **전환 전 예금의 금리를 판별 시점에 알 수 있나** — 이것만 답하면 `RATE_OVERRIDE` 변수 참조 건이 갈린다. 조사 결과 현행 유지 권고 (§15) | **팀 — io 명세 합의** |
| 17 | ~~override 가 지운 rule 기록~~ → **완료 (9/8).** `overridden_rules`. 조용히 우대금리를 지우고 ①~⑤ 를 통과할 수 있었다 | |

5번 항목은 이 프로젝트의 구조적 문제를 가리킨다.
**엔진은 금리를 계산하도록 설계됐는데, 이 상품의 핵심 혜택은 금리가 아니라 수수료 면제다.**
같은 뿌리의 문제를 이번 라운드에 세 번 만났다 —
`other_effects` 로 조용히 버려짐, ⑥ 이 그걸 못 잼, 배타 처리가 금리에만 적용됨.

---

## 부록 A. 9/8 이후 — 유실 구간 복구 기록 (v17 ~ v19)

2026-09-11 17:30(KST) 컨테이너가 재시작되면서 홈에 있던 저장소가 통째로 사라졌다.
v5 이후의 결과 파일, 실험 기록 §38~§50, 코드 변경분이 전부 여기 들어 있었다.
영구 볼륨에 남은 최신 산출물은 **9/3 의 `results_kbyy_v3.json` 하나**다.

> **이 부록의 출처는 대화 로그 하나뿐이다.**
> 아래 숫자는 원본 파일로 재확인할 수 없다. 재현하기 전까지 밖으로 인용하지 말 것.
> 코드 셀을 두지 않은 것도 같은 이유다 — 돌려서 확인할 수 있는 것이 없다.

그 사이 전제 둘이 달라졌다.

| | 본문 §1~§9 | 부록 A (v17~v19) |
|---|---|---|
| 회차 | 조문당 3회 | 조문당 **9회** |
| ⑥ 의 분모 | gold 15조문 한 덩어리 | 30조문을 **전사군 / 기권군** 으로 갈라 따로 잰다 |

§8.2 에서 "gold 15개 중 13개가 빈 배열이라 분모의 대부분이 모델이 가만히 있었나를
잰다" 고 적었던 것을, 빈 배열이 정답인 조문을 **기권군**으로 떼어 따로 재는 방식으로
풀었다. 나머지가 **전사군**이다. 라운드는 v19 까지 열일곱 번 돌았다.

### A.1 v19 — ⑤ 는 고쳐졌고 ⑥ 은 따라오지 않았다

지시형 문구를 넣자 ⑤ 가 움직였다. `S2-R12` 는 실행률 **67% → 100%** 다.
열려 있던 KeyError 경로가 닫혔고, **rule 6개짜리 형태와 KeyError 회차가
두 라운드 18회에서 예외 없이 1대1로 붙는다.** 상관이 아니라 대응이다.

그런데 ⑥ 은 따라오지 않았다. 30조문 중 움직인 것이 셋뿐이고, 가드가 뚫은 조문의
기여는 **0.04p** 다. **판정 불가**로 적었다.

지문 순위는 그대로다. v19 는 지문을 건드리지 않았으므로 **최고 지문은 여전히 v13**
이다. 다음 과녁은 **과생성**이고, 그 처방은 `prompt.py` 를 건드리므로 지문 해시가
갈리고 GPU 가 든다.

### A.2 군 평균이 조문 하나짜리였다

v18 → v19 의 이동을 조문별로 풀면 이렇다.

| 군 | 이동을 만든 것 | 크기 |
|---|---|---:|
| 전사군 | `S2-R01` 한 조문의 **아홉 회차 중 한 회차** | 0.96p |
| 기권군 | `S5-R03` **한 조문** (두 라운드 연속) | — |

`S2-R01` 은 rule 수도 같고 ⑤ 도 같다. 아홉 중 한 회차가 새 형태로 빠졌을 뿐인데
군 평균이 0.96p 파였다. **회차 하나의 무게가 2.2p** 다.

§9 의 5번과 같은 자리다. 거기서는 30조문 × 3회로 폭을 신호로 읽지 말라고 했는데,
9회로 늘린 뒤에도 조문 하나가 군 전체를 흔든다. **조문별로 풀기 전에는 군 평균을
처방의 효과로 읽지 않는다.**

### A.3 무출력 회차가 기권군에서 만점을 받고 있었다

잘리거나 파싱에 실패한 회차는 `output` 이 아예 없다. 그게 빈 배열로 흘러 기권군
채점에 들어갔다. **기권군은 빈 배열이 정답이므로 만점이 나온다.** 모델이 아무것도
못 내놓은 것과 낼 것이 없다고 판단한 것이 같은 점수를 받고 있었다.

세 라운드의 무출력 3 · 5 · 5 건이 **전부 `S5-R03` 하나**다.
§6.5 에서 분할 통제군으로 썼던 그 조문이다.

`gold.py score` 가 이제 두 갈래를 같이 찍는다. 지난 라운드 숫자가 전부 옛 갈래로
계산돼 있어서 기존 수치는 지우지 않았다.

| 기권군 ⑥ | 섞인 갈래 | 무출력 제외 |
|---|---:|---:|
| v17 | 93.3% | 92.4% |
| v18 | 94.6% | 94.1% |
| v19 | 94.0% | 92.6% |

전사군은 두 갈래가 같다. **이 수정은 전사군 판정을 하나도 건드리지 않는다.**

### A.4 인공물을 빼자 흔들림이 커졌다

v18 → v19 의 기권군 이동이 −0.6p 에서 **−1.5p** 로 커졌다. `S5-R03` 만 보면 더 분명하다.

| `S5-R03` | 9회 전체 | 무출력 제외 |
|---|---:|---:|
| v17 | 55.3% | 32.9% |
| v18 | 88.4% | 74.0% |
| v19 | 72.6% | 38.3% |

세 라운드에서 **41p** 를 오간다. 잘림이 점수를 올린 것도 맞지만, 그보다
**분산을 눌러 왔다**는 쪽이 중요하다.

그래서 기권군 문턱 **0.4p 는 못 쓴다.** 눌린 분산 위에서 잡힌 값이기 때문이다.
바닥은 **6.0p** 로 다시 명시했다. **실측 하나를 바닥으로 쓰면 안 된다** — 앞서
3.9p 를 바닥으로 적었다가 나중 라운드가 반증했다.

### A.5 회차를 빼면 조문 가중이 틀어진다 — 한 번 틀렸다 고쳤다

처음에는 프로필 가중으로 셌다. 그러자 회차를 뺀 만큼 **그 조문의 비중까지 줄었다.**
많이 잘리는 조문일수록 점수에 덜 반영되는 **두 번째 인공물**이 생긴다.
`S5-R03` 처럼 무출력이 몰린 조문에서 정확히 반대 방향으로 작동한다.

조문 가중으로 바꿨고 테스트로 고정했다. 테스트는 297개에서 **300개**가 됐다.

> 지표의 구멍은 점수를 틀리게 만드는 데 그치지 않는다.
> **분산을 눌러 판정을 거짓으로 세우기도 한다.** 기권군 문턱 0.4p 가 그렇게 생겼다.
>
> §6.2 의 참조표 중복, §8 의 `outcome()` 이 `other_effects` 를 안 보던 것에 이어
> 같은 종류로 **세 번째**다. §9 의 "세 번은 모델이 아니라 측정이 문제였다" 에
> 한 줄을 더 보태야 한다.

### A.6 조문 하나가 5,016초를 먹었다

마지막 라운드에서 29조문까지는 오히려 **84초 빨랐는데** 마지막 조문 하나가
**5,016초**를 먹었다. 원인 미상으로 적었다.

`run_local_v2.py` 의 진행 줄에 조문별 시간과 남의 GPU 점유를 추가했다. 누적에서
빼서 계산해야 했던 것과, GPU 경합 가설을 확인할 수 없었던 것이 둘 다 이 때문이다.

```
진행 13/30 [KBYY-S2-R12] 시도12 조문438s 예약16.4GB 여유13.8GB 타2.5GB  (1526s, ...)
```

**미확인** — 남은 기록 안에서 총 시간 배수가 2.1배와 6.9배로 두 번 다르게 나온다.
기준이 다른 것인지 오기인지 원본이 없어 판단이 안 된다.

### A.7 여기서 이어갈 것

| | 할 일 | GPU |
|---|---|---|
| 1 | 뺀 갈래로 **기권군 바닥 재측정** | 불필요 |
| 2 | **`S5-R03` 을 기권군에 남길지 결정** — 지표 설계 결정이라 따로 세운다 | 불필요 |
| 3 | **과생성 처방** — `prompt.py` 를 건드려 지문 해시가 갈린다 | 필요 |
| 4 | 조문 시간 이상 진단 — 계측은 붙었고 다음 실행에서 읽는다 | 필요 |

**다만 1·2 번이 전제하는 "기존 결과" 가 지금 없다.** v5 부터 v19 까지의 결과 파일과
`gold.py` · `run_local_v2.py` 의 수정분이 전부 유실됐다. 저장소를 복구하기 전에는
1번도 GPU 없이 되는 일이 아니다.

복구 순서는 이렇게 된다.

1. ~~저장소를 영구 볼륨에 다시 세운다~~ → **완료 (9/11).** `model/M2-2`
2. ~~모듈 12개를 다시 쓴다~~ → **완료 (9/11).** 테스트 425개 통과
3. ~~A.3 · A.5 의 채점 수정을 다시 넣고 테스트로 고정~~ → **완료.**
   `gold.clause_weighted` · `drop_empty_runs` · `tests/test_weighting.py`
4. ~~상품설명서 완본 구하기~~ → **완료.** 문서 저장소에 있었다 —
   `/workspace/database/outputs/KBKookmin/Deposit/20260818/text/`. 9,108자로 기록과 같다
5. ~~gold 초안~~ → **완료.** KBYY 15 + KBONE 15 = **30개.** 전부 `reviewed: false`
6. ~~라운드를 v1 부터 다시 시작~~ → **v1~v9 진행 중**

## 다시 시작한 라운드가 알아낸 것

번호를 v1 부터 다시 셌다. 유실된 v13~v19 의 지문을 복원할 방법이 없어
노트북에 남은 설계 원칙으로 새로 썼기 때문이다. 코퍼스는 두 상품 46조문이다
(KB Young Youth 통장 31 + KB 국민ONE통장 20, 본문이 같은 5조문 제거).

**형식 관문은 크게 올랐다** (v1 → v7, 138 레코드 동일 조건).

| 지표 | v1 | v7 |
|---|---:|---:|
| ① JSON 파싱 | 76.1% | 99.3% |
| ② 스키마 적합 | 71.0% | 99.3% |
| ③ 카탈로그 준수 | 52.2% | 77.5% |
| ④ quote 일치 | 50.7% | 73.9% |
| ⑤ 실행 가능 | 47.8% | 70.3% |

먹은 처방은 셋이다. **few-shot 을 들여쓴 JSON 으로**(압축본은 모델이 중괄호
깊이를 놓친다), **재시도 프롬프트의 출력도 들여쓰기**, **few-shot 조문을 코퍼스와
같은 조판으로**(대괄호·줄머리 기호를 그대로 안고 인용하는 장면이 없었다).
넷째로 **재시도 중 가장 나은 시도를 채택**하게 해서 자기수정이 결과를 망가뜨리는
일을 0 으로 만들었다.

**그런데 이 이득이 뜻으로 이어지지 않았다.** 감사를 붙여 확인했다.

> `KBYY-S1-P01` 은 **숫자가 하나도 없는 두 줄짜리 조문**인데 생성물이 우대금리
> `0.5 %p` 와 면제횟수 `100` 을 만들어 내고 **관문 다섯을 전부 통과했다.**

①~⑤ 중 어느 것도 **값이 어디서 왔는지 묻지 않는다.** ⑥ 도 못 잡는다 —
gold 가 빈 배열인 조문에서는 창작 rule 이 어느 프로필에서도 발화하지 않으면
실행 결과가 빈 배열과 같기 때문이다.

그래서 뜻을 재는 축을 넷 세웠다.

| 축 | 무엇 | v1 | v7 |
|---|---|---:|---:|
| ⑨ 근거 | 생성물의 수가 조문에 있나 | 34.8% | **58.7%** |
| ⑩ 합성 | 조문을 **합쳐** 낸 금리가 gold 와 같나 | 25.0% | 28.1% |
| ⑪ 뒷받침 | 인용문이 그 rule 을 받치나 (하한) | — | 8.4% |
| ⑫ 선언 | 빈 배열/UNSUPPORTED/정상 판정이 맞나 | 30.0% | 40.0% |

**⑨ 가 형식 관문과 정확히 반대로 갔다.** 형식이 +23p 오르는 동안 숫자 발명이
+24p 늘었다. 모델이 유효한 JSON 을 더 잘 만들게 되면서 빈칸을 그럴듯한 수로
채우는 일도 더 잘하게 됐다.

⑩ 이 가장 중요하다. ⑥ 은 조문을 **하나씩 고립시켜** 재므로,
`RATE_OVERRIDE 0.1` 처럼 **합성할 때만 드러나는 파괴**를 못 본다.
이 부록의 gold 노트가 "특별금리 1.9 를 통째로 지운다" 고 미리 경고해 둔 형태인데
⑥ 은 100점을 준다.

**이 부록이 지목했던 과녁이 맞았다.** A.1 이 "다음 과녁은 과생성" 이라고 적었는데,
⑫ 로 재 보니 만들 것이 없는데 만드는 것이 11건, 반대 방향이 1건이다.
축이 없어서 그동안 못 봤을 뿐이다.

자세한 것은 `M2-2/README.md` 와 `M2-2/ROUNDS.md` 에 있다.

### A.8 v9 ~ v20 — 지문으로는 못 푸는 벽

라운드를 계속 돌려 v20 까지 왔다. 형식 관문은 v7 에서 거의 천장을 쳤고,
그 뒤 라운드는 전부 **뜻**을 겨눴다. 결론 다섯 가지를 적는다.

#### 1. 병목은 지문이 아니라 카탈로그다

v15 에서 `DISQUALIFY` 오용을 예시로 막았다. 그 사용은 13 → 8 로 줄었는데
거래중지 조문이 `LIMIT_AMOUNT` 로 옮겨가고 **지어낸 코드가 1 → 7 로 늘었다.**

**막으면 다른 도구로 샌다.** 옮길 자리가 없는 것이 원인이기 때문이다.
gold 46조문 중 **30조문에 표현 불가 rule 이 있다.**

#### 2. 예시 자리는 유한하고 맞바꿀 수 없다

v9 와 v18 에서 두 번 확인했다. 새 장면을 넣으려고 빈 배열 예시를 빼자
⑦ 이 30.4% → 8.7% 로 떨어지고 과생성이 15건이 됐다. 둘 다 되돌렸다.

#### 3. 설명 줄은 어휘를 옮기고 뜻은 못 옮긴다 (v17 · v18 · v19)

같은 조문(거래중지)에 세 가지 방법을 써 봤다.

| 방법 | 생성물 | |
|---|---|---|
| v17 이름만 준다 | `term_months >= 12` | 가입기간을 무거래기간으로 오인 |
| v18 장면을 보여준다 | `count_events(WITHDRAWAL) == 0` | **맞다** |
| v19 어휘에 설명 줄 | `count_events(WITHDRAWAL) >= 0` | **언제나 참이다** |

표면 지표로는 v19 가 v18 의 7할을 했다 (해당 이름을 쓴 rule 7 대 10).
**뜻으로는 0할이다.** `>= 0` 은 셀 수 없는 값이 0 이상인지 묻는 헛말이라,
잔액이 적은 **모든 정상 계좌**가 거래중지된다. 뜻이 정반대로 뒤집힌다.

표면만 봤다면 "예시 자리 없이 카탈로그를 넓힐 수 있다" 고 결론 냈을 것이다.

#### 4. 그래서 축을 하나 더 세웠다 — ⑭ 헛조건

**조건이 붙어 있는데 어떤 문맥에서도 참인 것**을 센다. 관문 다섯이 전부
통과시키고 ⑫ 도 못 본다. 판정 자체는 맞기 때문이다.

이 축을 만들면서 내가 낸 수치를 두 번 정정했다.

* 창을 `$window` 로 참조하는 조건에 바인딩을 안 넘겨 **검사에서 통째로 빠졌다.**
  이 코퍼스에서 가장 흔한 형태라 축이 거의 아무것도 못 보고 있었다.
* 고치고 나서 "조건의 15~30%가 장식" 이라고 적었는데 **여덟 배 부풀린 값**이었다.
  프로필이 변수를 안 흔들어서 참인 것과 어떤 문맥에서도 참인 것을 섞어 셌다.
  카탈로그 정의역을 훑는 문맥 400개로 반례를 찾아 갈랐다. 실제 값은 v18 기준 1.9% 다.

#### 5. 카탈로그 확장 우선순위가 뒤집혔다

확장 안건이 우선순위를 **항목을 언급한 조문 수**로 매기고 있었다. 틀린 셈법이다.
한 조문이 두 가지를 동시에 요구하면 하나만 넣어서는 **아무것도 안 풀린다.**

| 항목 | 옛 순위 | 단독으로 푸는 조문 |
|---|---|---|
| 제3자 서비스 제공 effect | 10위 (꼴찌) | **3** |
| 출금 이벤트 | 2위 | **0** |

v17 에서 출금 이벤트를 실제로 넣어 확인했다. 풀린 조문은 0개다 — 거래중지
조문이 무거래기간을 "입출금 거래 **또는 명의변경 거래**" 로 정의하기 때문이다.

`M2-2/blockers.py` 가 이 셈을 한다. 다만 gold 가 전부 `reviewed: false` 인
초안이라 **사람 검토가 이 순위를 바꿀 수 있다.**

#### 지금 사람 손이 필요한 것

1. **gold 46개 검토.** 모든 축이 여기에 걸려 있다. 확장 순위도 여기서 나온다
2. **카탈로그 확장 결정** — `M2-2/CATALOG_PROPOSAL.md`. io 명세 합의가 필요하다

자세한 것은 `M2-2/ROUNDS.md` 와 `M2-2/EXPERIMENT_v*.md` 에 있다.